# Symmetric Lanczos — Interactive Experiments

Explore Lanczos tridiagonalization for **real symmetric** and **complex
Hermitian** systems. Compare Ritz values to exact eigenvalues, assess
Ritz-pair residuals, and examine the optional Galerkin approximation.

Choose the system, Krylov dimension, spectral spread, and
reorthogonalization setting, then click **Run experiment**. Test systems
are generated with a fixed seed and known exact solutions.


In [2]:
from pathlib import Path
import sys
import time

# Locate the repository root for local Jupyter or Binder execution.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Launch Jupyter from within the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from krylov_methods.notebooks.notebook_utils import (
    spd_problem, hermitian_pd_problem, symmetric_indefinite_problem,
    residual_plot, error_plot, report,
    ritz_spectrum_plot, ritz_residual_plot,
)
from krylov_methods.symmetric_lanczos_solver.Lanczos import lanczos_tridiagonalize
from krylov_methods.symmetric_lanczos_solver.galerkin import lanczos_galerkin


def test_system(kind, n, kappa):
    """Create a reproducible Hermitian system with known exact solution."""
    if kind == "real":
        return spd_problem(n, kappa)
    if kind == "complex":
        return hermitian_pd_problem(n, kappa)
    if kind == "indefinite":
        return symmetric_indefinite_problem(n, kappa)
    raise ValueError(f"Unknown system type: {kind}")


## 1. Mathematical foundation and interactive experiment

For a Hermitian matrix $A$ and normalized starting vector $v_1$, Lanczos
constructs $V_m=[v_1,\ldots,v_m]$ and a real symmetric tridiagonal
matrix $T_m$ satisfying, in exact arithmetic,

<p align="center">$\displaystyle AV_m=V_mT_m+\beta_m v_{m+1}e_m^T.$</p>

The eigenvalues $\theta_i$ of $T_m$ are **Ritz values**. If
$T_my_i=\theta_i y_i$, then $u_i=V_my_i$ is a Ritz vector. A useful measure
of accuracy is the Ritz-pair residual

<p align="center">$\displaystyle \|Au_i-\theta_i u_i\|_2.$</p>

For a linear system $Ax=b$, the optional **Galerkin approximation** is
$x_m=x_0+V_my_m$, with $T_my_m=\gamma e_1$ and
$\gamma=\|b-Ax_0\|_2$. It enforces

<p align="center">$\displaystyle V_m^H(b-Ax_m)=0.$</p>

The Galerkin solution may not exist when $T_m$ is singular, especially for
indefinite matrices. Lanczos itself remains useful for spectral analysis.


In [4]:
system = widgets.Dropdown(
    options=[
        ("Real SPD", "real"),
        ("Complex Hermitian PD", "complex"),
        ("Real symmetric indefinite", "indefinite"),
    ],
    value="real", description="System",
)
size = widgets.IntSlider(value=30, min=5, max=100, step=5, description="Size n")
dimension = widgets.IntSlider(value=15, min=1, max=100, step=1, description="Krylov m")
condition = widgets.FloatLogSlider(
    value=1e3, base=10, min=1, max=6, step=0.5, description="Spectral spread"
)
reorth = widgets.Checkbox(value=True, description="Full reorthogonalization")
tol = widgets.FloatLogSlider(value=1e-10, base=10, min=-12, max=-6,
                             step=1, description="Galerkin tol")
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        A, b, x_true = test_system(system.value, size.value, condition.value)
        m = min(size.value, dimension.value)
        eigs = np.linalg.eigvalsh(A)
        print("SYSTEM INFORMATION")
        labels = {"real": "Real SPD", "complex": "Complex Hermitian PD",
                  "indefinite": "Real symmetric indefinite"}
        print(f"Matrix: {labels[system.value]}; dimension={size.value}; dtype={A.dtype}")
        print(f"||A-Aᴴ||₂={np.linalg.norm(A-A.conj().T, ord=2):.3e}")
        print(f"Eigenvalue interval=[{eigs[0]:.3e}, {eigs[-1]:.3e}]")
        print(f"Number of negative eigenvalues={np.count_nonzero(eigs < 0)}")
        print(f"Condition number κ₂(A)={np.linalg.cond(A):.3e}")
        print(f"||b||₂={np.linalg.norm(b):.3e}; ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"Requested Krylov dimension={m}; reorthogonalize={reorth.value}")
        print(f"Galerkin stopping threshold={tol.value:.3e} (absolute)\n")

        start = time.perf_counter()
        lanczos = lanczos_tridiagonalize(A, b, m=m, reorthogonalize=reorth.value)
        time_lanczos = time.perf_counter() - start
        V, T = lanczos.basis, lanczos.tridiagonal
        orth = np.linalg.norm(V.conj().T @ V - np.eye(lanczos.iterations))
        projection = np.linalg.norm(V.conj().T @ A @ V - T)
        print("LANCZOS TRIDIAGONALIZATION")
        print(f"Achieved dimension={lanczos.iterations}; status={lanczos.message}")
        print(f"Orthogonality error ||VᴴV-I||₂={orth:.3e}")
        print(f"Projection error ||VᴴAV-T||₂={projection:.3e}")
        print(f"Tridiagonalization time={time_lanczos:.6f} s")

        # Evaluate Ritz-pair quality via the shared plotting helper.
        theta = np.linalg.eigvalsh(T)
        print("Ritz values:", np.array2string(theta, precision=5))
        ritz_residuals = np.linalg.norm(
            A @ (V @ np.linalg.eigh(T)[1]) -
            (V @ np.linalg.eigh(T)[1]) * theta, axis=0
        )
        print(f"Max Ritz-pair residual={np.max(ritz_residuals):.3e}")

        start = time.perf_counter()
        projected = lanczos_galerkin(A, b, m=m, tol=tol.value,
                                     reorthogonalize=reorth.value)
        time_galerkin = time.perf_counter() - start
        print("\nGALERKIN APPROXIMATION")
        if projected.solution is not None:
            report("Lanczos Galerkin", projected.solution, x_true, b, A,
                   lanczos.iterations, projected.converged)
            print(f"||b-Ax||₂={projected.residual_norm:.3e}; "
                  f"relative error={np.linalg.norm(projected.solution-x_true)/np.linalg.norm(x_true):.3e}")
        else:
            print("No projected solution:", projected.message)
        print(f"Galerkin call time={time_galerkin:.6f} s")

        ritz_spectrum_plot(A, theta)
        ritz_residual_plot(A, V, T)


button.on_click(run)
display(widgets.VBox([
    system, size, dimension, condition, reorth, tol, button, output,
]))


## 2. Numerical interpretation

- **Ritz values** are approximations to eigenvalues; accuracy is better
  assessed with Ritz-pair residuals than by visual proximity alone.
- **Orthogonality:** Lanczos has a short three-term recurrence, but rounding
  can cause loss of basis orthogonality. Full reorthogonalization can help,
  at additional memory and computation cost.
- **Galerkin residual vs error:** A small residual need not imply a small
  forward error for ill-conditioned systems. Both quantities are reported.
- **Indefinite systems:** The projected Galerkin matrix may become singular.
  MINRES is generally preferable for robust symmetric-indefinite solves.

Unlike the CG, GMRES, and BiCGSTAB solvers, this Lanczos module has **no
preconditioner selection**. Preconditioning a Hermitian Lanczos process
requires additional symmetry-preserving structure.

## 3. Explore Krylov dimension and reorthogonalization

The experiment below compares orthogonality errors, true Galerkin residuals,
and relative solution errors across several Krylov dimensions. It runs
both the classical recurrence and full reorthogonalization on the same
test system.


In [6]:
study_button = widgets.Button(
    description="Explore dimension and orthogonality", button_style="info"
)
study_output = widgets.Output()


def run_study(_):
    with study_output:
        clear_output(wait=True)
        A, b, x_true = test_system(system.value, size.value, condition.value)
        # A few Krylov dimensions keep this interactive experiment responsive.
        candidates = sorted(set([
            1, min(3, size.value), min(5, size.value),
            min(10, size.value), min(15, size.value),
            min(20, size.value), size.value,
        ]))
        results = {False: {"orth": [], "res": [], "error": []},
                   True: {"orth": [], "res": [], "error": []}}
        for use_reorth in (False, True):
            for requested in candidates:
                L = lanczos_tridiagonalize(A, b, m=requested,
                                          reorthogonalize=use_reorth)
                Q = L.basis
                orth_err = np.linalg.norm(Q.conj().T @ Q - np.eye(L.iterations))
                projected = lanczos_galerkin(A, b, m=requested, tol=1e-12,
                                             reorthogonalize=use_reorth)
                results[use_reorth]["orth"].append(orth_err)
                results[use_reorth]["res"].append(
                    np.nan if projected.residual_norm is None else projected.residual_norm
                )
                results[use_reorth]["error"].append(
                    np.nan if projected.solution is None else
                    np.linalg.norm(projected.solution-x_true)/np.linalg.norm(x_true)
                )
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
        for use_reorth, color in ((False, "tab:orange"), (True, "tab:blue")):
            label = "Full reorthogonalization" if use_reorth else "Three-term only"
            for ax, key in zip(axes, ("orth", "res", "error")):
                y = np.asarray(results[use_reorth][key], dtype=float)
                ax.semilogy(candidates, np.where(np.isfinite(y),
                            np.maximum(y, np.finfo(float).tiny), np.nan),
                            marker="o", label=label, color=color)
        for ax, ylabel in zip(axes, (
            r"$\|V_m^H V_m-I\|_2$",
            r"$\|b-Ax_m\|_2$",
            r"$\|x_m-x^*\|_2/\|x^*\|_2$",
        )):
            ax.set(xlabel="Requested Krylov dimension m", ylabel=ylabel)
            ax.grid(True, which="both", alpha=0.25)
            ax.margins(y=0.12)
        axes[0].legend()
        fig.suptitle("Lanczos: Krylov dimension and reorthogonalization")
        fig.tight_layout()
        plt.show()
        print("NaN or missing curve points indicate an undefined singular projected solve.")
        print("Full reorthogonalization can improve orthogonality, at extra cost.")


study_button.on_click(run_study)
display(widgets.VBox([study_button, study_output]))


## 4. Solving a linear system with Lanczos–Galerkin

We now test the method on the linear system $Ax=b$

with a known exact solution $x^*$ and a zero initial guess. After constructing
the Lanczos basis $V_m$ from $r_0=b-Ax_0$, the Galerkin approximation is

<p align="center">$\displaystyle x_m=x_0+V_my_m,\qquad T_my_m=\|r_0\|_2e_1.$</p>

We compare the Lanczos–Galerkin approximation against the exact direct solve
(`numpy.linalg.solve`) and a suitable SciPy iterative solver:

- **CG** for real SPD and complex Hermitian positive-definite systems.
- **MINRES** for real symmetric-indefinite systems.

We report the true residual $\|b-Ax_m\|_2$ and relative forward error
$\|x_m-x^*\|_2/\|x^*\|_2$. Unlike CG and MINRES, the standalone projected
Galerkin method may be undefined when the projected $T_m$ is singular.

The convergence curves below plot these quantities against the **Krylov
dimension** $m$, not against elapsed time. A smaller residual need not imply
a proportionally smaller solution error when the system is ill-conditioned.


In [8]:
from scipy.sparse.linalg import cg as scipy_cg, minres as scipy_minres

solve_button = widgets.Button(description="Solve Ax=b and compare", button_style="success")
solve_output = widgets.Output()


def solve_system(_):
    with solve_output:
        clear_output(wait=True)
        A, b, x_true = test_system(system.value, size.value, condition.value)
        n = A.shape[0]
        selected_m = min(dimension.value, n)
        x_direct = np.linalg.solve(A, b)
        exact_err = np.linalg.norm(x_direct - x_true) / np.linalg.norm(x_true)
        print("LINEAR SYSTEM")
        print(f"System={system.value}, dimension={n}, cond₂(A)={np.linalg.cond(A):.3e}")
        print(f"||b||₂={np.linalg.norm(b):.3e}; ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"Direct solve relative error={exact_err:.3e}")
        print(f"Selected Krylov dimension m={selected_m}; absolute Galerkin tolerance={tol.value:.1e}\n")

        # The projected method is tested both at the selected dimension and
        # at the full dimension to illustrate finite-dimensional convergence.
        for label, m in (("Selected dimension", selected_m), ("Full dimension", n)):
            start = time.perf_counter()
            result = lanczos_galerkin(A, b, m=m, tol=tol.value,
                                      reorthogonalize=reorth.value)
            elapsed = time.perf_counter() - start
            if result.solution is None:
                print(f"Lanczos–Galerkin ({label}, m={m}): {result.message}")
            else:
                report(f"Galerkin m={m}", result.solution, x_true, b, A,
                       result.lanczos.iterations, result.converged)
                print(f"  Time={elapsed:.6f} s; {result.message}")

        # SciPy's relative tolerance is adjusted to roughly match the
        # absolute true-residual threshold used by Lanczos–Galerkin.
        scipy_rtol = tol.value / max(np.linalg.norm(b), np.finfo(float).tiny)
        scipy_history = [np.linalg.norm(b)]
        def cb(xk):
            scipy_history.append(np.linalg.norm(b - A @ xk))
        start = time.perf_counter()
        if system.value == "indefinite":
            scipy_x, info = scipy_minres(A, b, rtol=scipy_rtol,
                                        maxiter=max(5*n, 100), callback=cb)
            scipy_label = "SciPy MINRES"
        else:
            scipy_x, info = scipy_cg(A, b, rtol=scipy_rtol, atol=0.0,
                                     maxiter=max(5*n, 100), callback=cb)
            scipy_label = "SciPy CG"
        elapsed = time.perf_counter() - start
        report(scipy_label, scipy_x, x_true, b, A,
               len(scipy_history)-1, info == 0)
        print(f"  Time={elapsed:.6f} s; SciPy info={info}\n")

        # Sample at most ~14 dimensions to keep interactive use responsive.
        sample = sorted(set([1, selected_m, n] +
                            list(range(1, n+1, max(1, n // 12)))))
        dims, residual_norms, relative_errors = [], [], []
        for m in sample:
            result = lanczos_galerkin(A, b, m=m, tol=tol.value,
                                      reorthogonalize=reorth.value)
            dims.append(m)
            if result.solution is None:
                residual_norms.append(np.nan)
                relative_errors.append(np.nan)
            else:
                residual_norms.append(np.linalg.norm(b-A@result.solution))
                relative_errors.append(
                    np.linalg.norm(result.solution-x_true)/np.linalg.norm(x_true)
                )
        dims = np.asarray(dims)
        residual_norms = np.asarray(residual_norms)
        relative_errors = np.asarray(relative_errors)
        # The shared plot helpers reject NaN histories; show undefined points
        # as gaps on the axes while preserving their true m coordinates.
        def plot_finite(ax, values, label):
            good = np.isfinite(values)
            ax.semilogy(dims[good], np.maximum(values[good], np.finfo(float).tiny),
                        marker="o", linewidth=1.5, label=label)

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        plot_finite(axes[0], residual_norms, "Lanczos–Galerkin")
        axes[0].axhline(tol.value, color="black", linestyle="--",
                        label="Galerkin tolerance")
        axes[0].set(ylabel=r"True residual $\|b-Ax_m\|_2$",
                    title="Residual versus Krylov dimension")
        plot_finite(axes[1], relative_errors, "Lanczos–Galerkin")
        axes[1].set(ylabel=r"Relative error $\|x_m-x^*\|_2/\|x^*\|_2$",
                    title="Forward error versus Krylov dimension")
        for ax in axes:
            ax.set_xlabel("Krylov dimension m")
            ax.grid(True, which="both", alpha=0.25)
            ax.legend()
            ax.margins(y=0.12)
        fig.tight_layout()
        plt.show()
        if np.any(~np.isfinite(residual_norms)):
            print("Some Krylov dimensions produced singular projected systems; "
                  "their plot points are omitted.")
        print("Interpretation: compare the true residual and relative solution error; "
              "the relationship depends on the conditioning of A.")


solve_button.on_click(solve_system)
display(widgets.VBox([solve_button, solve_output]))
